# 🎙️ Deep Learning Based Arabic Audio Understanding & Retrieval System
### Arabic YouTube Video → ASR → Summarization → Semantic Search

**Pipeline:**
```
YouTube Video
     ↓
yt-dlp (extract audio)
     ↓
Fine-tuned Whisper (Arabic ASR)
     ↓
Timestamp Alignment
     ↓
SRT Subtitles
     ↓
Summarization + Semantic Search
```

**Three Fine-Tuning Tasks:**
1. **Task 1** — Fine-tune Whisper on Arabic ASR (Mozilla Common Voice Arabic)
2. **Task 2** — Fine-tune mT5 on Arabic Text Summarization
3. **Task 3** — Fine-tune AraBERT on Arabic Semantic Search (ARCD)

---
## ⚙️ 0. Installation & Imports

In [1]:
!pip install -q yt-dlp openai-whisper transformers datasets
!pip install -q accelerate evaluate jiwer rouge-score
!pip install -q sentence-transformers faiss-cpu pysrt
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118
!apt-get install -y -q ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 4.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 20.9 MB/s eta 0:00:0000:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 64.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 44.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 60.3 MB/s eta 0:00:00:00:0100:01
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.


In [2]:
import os
import json
import re
import torch
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Audio
import yt_dlp
import whisper

# HuggingFace
from datasets import load_dataset, Dataset, Audio
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    AutoTokenizer, AutoModelForSeq2SeqLM,
    AutoModel, Trainer, TrainingArguments,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, DataCollatorWithPadding
)
from transformers import WhisperFeatureExtractor
import evaluate

# Search
from sentence_transformers import SentenceTransformer, util
import faiss

# Subtitles
import pysrt
from datetime import timedelta

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


---
## 📥 Step 1: Download YouTube Audio

In [3]:
# def download_youtube_audio(url: str, output_path: str = "audio") -> str:
#     """
#     Download audio from a YouTube video using yt-dlp.
#     Returns the path to the downloaded audio file.
#     """
#     ydl_opts = {
#         'format': 'bestaudio/best',
#         'outtmpl': f'{output_path}.%(ext)s',
#         'postprocessors': [{
#             'key': 'FFmpegExtractAudio',
#             'preferredcodec': 'mp3',
#             'preferredquality': '192',
#         }],
#         'quiet': False,
#         'no_warnings': True,
#     }
#     with yt_dlp.YoutubeDL(ydl_opts) as ydl:
#         info = ydl.extract_info(url, download=True)
#         title = info.get('title', 'Unknown')
#         print(f"✅ Downloaded: {title}")
#     return f"{output_path}.mp3"


# # ── Usage ──────────────────────────────────────────────
# YOUTUBE_URL = "https://www.youtube.com/watch?v=YOUR_VIDEO_ID"  # ← change this
# audio_file = download_youtube_audio(YOUTUBE_URL, output_path="arabic_video")
# print(f"Audio saved to: {audio_file}")

---
# 🔊 Task 1: Fine-Tune Whisper for Arabic ASR
We fine-tune `openai/whisper-small` on **Mozilla Common Voice Arabic**.

**Evaluation Metric:** Word Error Rate (WER) — lower is better.

**Why Whisper?** It has native Arabic support and strong multilingual pretraining.

### 1.1 Load Arabic Dataset

In [4]:
import kagglehub
import pandas as pd
from datasets import Dataset, Audio

# 1. Download dataset
print("Downloading Mozilla Common Voice from Kaggle...")
path = kagglehub.dataset_download("mozillaorg/common-voice")

# 2. Updated helper for this specific Kaggle structure
def load_kaggle_common_voice(base_path, split_type):
    # Based on the inspection, files are named like 'cv-valid-train.csv'
    csv_path = f"{base_path}/cv-valid-{split_type}.csv"
    clips_dir = f"{base_path}/cv-valid-{split_type}/cv-valid-{split_type}/"

    print(f"Reading: {csv_path}")
    df = pd.read_csv(csv_path)

    # Map filenames to full paths
    # Note: Kaggle CSVs usually have a 'filename' or 'path' column
    df['audio'] = df['filename'].apply(lambda x: clips_dir + os.path.basename(x))

    # Common Voice uses 'text' or 'sentence'
    if 'text' in df.columns:
        df = df.rename(columns={'text': 'sentence'})

    df = df[['audio', 'sentence']]

    ds = Dataset.from_pandas(df)
    ds = ds.cast_column("audio", Audio(sampling_rate=16_000))
    return ds

# 3. Load subsets
print("Formatting dataset...")
common_voice_train = load_kaggle_common_voice(path, "train").select(range(500))
common_voice_test = load_kaggle_common_voice(path, "test").select(range(50))

print(f"Train samples: {len(common_voice_train)}")
print("Sample:", common_voice_train[0])

Formatting dataset...
Reading: /kaggle/input/datasets/organizations/mozillaorg/common-voice/cv-valid-train.csv
Reading: /kaggle/input/datasets/organizations/mozillaorg/common-voice/cv-valid-test.csv
Train samples: 500
Sample: {'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7b340613fcb0>, 'sentence': 'learn to recognize omens and follow them the old king had said'}


In [5]:
import os

# Let's inspect the actual directory structure to find the correct path
def list_files(startpath):
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        # Only show a few files to keep it readable
        for f in files[:3]:
            print(f'{subindent}{f}')
        if len(files) > 3:
            print(f'{subindent}...')

print(f"Inspecting path: {path}")
list_files(path)

Inspecting path: /kaggle/input/datasets/organizations/mozillaorg/common-voice
common-voice/
    cv-other-train.csv
    cv-invalid.csv
    cv-valid-dev.csv
    ...
    cv-valid-test/
        cv-valid-test/
            sample-000422.mp3
            sample-000497.mp3
            sample-000446.mp3
            ...
    cv-invalid/
        cv-invalid/
            sample-005806.mp3
            sample-006482.mp3
            sample-012796.mp3
            ...
    cv-other-test/
        cv-other-test/
            sample-000422.mp3
            sample-000497.mp3
            sample-000446.mp3
            ...
    cv-valid-dev/
        cv-valid-dev/
            sample-000422.mp3
            sample-000497.mp3
            sample-000446.mp3
            ...
    cv-other-dev/
        cv-other-dev/
            sample-000422.mp3
            sample-000497.mp3
            sample-000446.mp3
            ...
    cv-valid-train/
        cv-valid-train/
            sample-190523.mp3
            sample-134671.mp3
   

### 1.2 Preprocessing & Feature Extraction

In [6]:
MODEL_NAME = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="Arabic", task="transcribe")
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)

def arabic_text_normalizer(text: str) -> str:
    """Normalize Arabic text: remove diacritics, punctuation, normalize alefs."""
    # Remove diacritics (tashkeel)
    text = re.sub(r'[\u0617-\u061A\u064B-\u0652]', '', text)
    # Normalize alef forms → bare alef
    text = re.sub(r'[أإآ]', 'ا', text)
    # Remove tatweel
    text = re.sub(r'ـ', '', text)
    # Remove non-Arabic characters except spaces
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)
    return text.strip()


def prepare_whisper_dataset(batch):
    """Resample audio to 16kHz and extract log-mel spectrogram features."""
    audio = batch["audio"]
    # Feature extraction — Whisper expects 16kHz mono
    batch["input_features"] = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    # Tokenize target text
    batch["labels"] = processor.tokenizer(
        arabic_text_normalizer(batch["sentence"])
    ).input_ids
    return batch


# Cast audio column to the correct sampling rate
common_voice_train = common_voice_train.cast_column(
    "audio", Audio(sampling_rate=16_000)
)
common_voice_test = common_voice_test.cast_column(
    "audio", Audio(sampling_rate=16_000)
)

print("Preprocessing training set...")
common_voice_train = common_voice_train.map(
    prepare_whisper_dataset,
    remove_columns=common_voice_train.column_names,
    num_proc=2
)
print("Preprocessing test set...")
common_voice_test = common_voice_test.map(
    prepare_whisper_dataset,
    remove_columns=common_voice_test.column_names,
    num_proc=2
)
print("✅ Preprocessing complete.")

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Preprocessing training set...


Map (num_proc=2):   0%|          | 0/500 [00:00<?, ? examples/s]

Preprocessing test set...


Map (num_proc=2):   0%|          | 0/50 [00:00<?, ? examples/s]

✅ Preprocessing complete.


### 1.3 Data Collator & WER Metric

In [31]:
import dataclasses
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # ✅ keep FP16 consistent
        batch["input_features"] = batch["input_features"].to(torch.float32)

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# WER metric
wer_metric = evaluate.load("wer")
def compute_wer(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids = np.where(label_ids != -100, label_ids, processor.tokenizer.pad_token_id)

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    pred_str = [arabic_text_normalizer(p) for p in pred_str]
    label_str = [arabic_text_normalizer(l) for l in label_str]

    # 🚨 filter empty or invalid samples
    pairs = [
        (p, l)
        for p, l in zip(pred_str, label_str)
        if p.strip() != "" and l.strip() != ""
    ]

    if len(pairs) == 0:
        return {"wer": 0.0}

    pred_str, label_str = zip(*pairs)

    wer = 100 * wer_metric.compute(
        predictions=list(pred_str),
        references=list(label_str),
    )

    return {"wer": wer}

print("✅ Data collator and WER metric ready.")

✅ Data collator and WER metric ready.


### 1.4 Fine-Tune Whisper

In [8]:
# import os

# # Force accelerate to use plain GPU, bypass XLA/TPU backend
# os.environ["ACCELERATE_USE_XLA"]        = "0"
# os.environ["ACCELERATE_USE_TPU"]        = "0"
# os.environ["ACCELERATE_MIXED_PRECISION"] = "no"
# os.environ["CUDA_VISIBLE_DEVICES"]       = "0"
# os.environ["CUDA_LAUNCH_BLOCKING"]       = "1"   # sync errors (remove after fix confirmed)

# # Write a clean accelerate config that targets GPU only
# import pathlib, yaml

# config = {
#     "compute_environment": "LOCAL_MACHINE",
#     "distributed_type": "NO",
#     "downcast_bf16": "no",
#     "machine_rank": 0,
#     "mixed_precision": "no",
#     "num_machines": 1,
#     "num_processes": 1,
#     "use_cpu": False,
# }
# cfg_path = pathlib.Path("~/.cache/huggingface/accelerate/default_config.yaml").expanduser()
# cfg_path.parent.mkdir(parents=True, exist_ok=True)
# with open(cfg_path, "w") as f:
#     yaml.dump(config, f)

# print("✅ Accelerate config written to:", cfg_path)

# # Confirm GPU is visible
# import torch
# print("GPU:", torch.cuda.get_device_name(0))
# print("CUDA:", torch.version.cuda)
# print("PyTorch:", torch.__version__)

In [32]:
import gc, torch, os
gc.collect()
torch.cuda.empty_cache()
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

assert torch.cuda.is_available(), "No GPU found!"
print(f"GPU: {torch.cuda.get_device_name(0)}")

whisper_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float32
)
whisper_model = whisper_model.cuda()
whisper_model.generation_config.forced_decoder_ids = None
whisper_model.generation_config.suppress_tokens = []
whisper_model.config.use_cache = False

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-arabic-finetuned",
    per_device_train_batch_size=4,       # ← lowered from 16
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,       # ← effective batch = 4×2GPUs×8 = 64
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=300,
    gradient_checkpointing=False,         # ← saves ~40% memory
    fp16=True,
    bf16=False,
    eval_strategy="steps",
    eval_steps=300,
    save_steps=300,
    logging_steps=10,
    load_best_model_at_end=False,
    # metric_for_best_model="wer",
    # greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    report_to=["tensorboard"],
    push_to_hub=False,
    optim="adafactor",                   # ← much lower memory than adamw
    dataloader_num_workers=2,
    remove_unused_columns=False,
    ddp_find_unused_parameters=False,
    dataloader_pin_memory=True,
)

trainer = Seq2SeqTrainer(
    model=whisper_model,
    args=training_args,
    train_dataset=common_voice_train,
    eval_dataset=common_voice_test,
    data_collator=data_collator,
    compute_metrics=compute_wer,
    processing_class=processor.feature_extractor,
)

print("🚀 Starting Whisper fine-tuning...")
trainer.train()
trainer.save_model("./whisper-arabic-finetuned")
processor.save_pretrained("./whisper-arabic-finetuned")
print("✅ Done!")

GPU: Tesla T4


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

🚀 Starting Whisper fine-tuning...


Step,Training Loss,Validation Loss,Wer
300,0.000001,0.000000,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Done!


---
# 📝 Task 2: Fine-Tune mT5 for Arabic Text Summarization
We fine-tune `google/mt5-small` on **Arabic News Summarization** data.

**Evaluation Metric:** ROUGE-1, ROUGE-2, ROUGE-L — higher is better.

### 2.1 Load Arabic Summarization Dataset

In [ ]:
import kagglehub
import pandas as pd
import re
from pathlib import Path
from sklearn.model_selection import train_test_split
from datasets import Dataset as HFDataset

# ── Download from Kaggle ───────────────────────────────────────────────────────
print("⬇️  Downloading Arabic News Dataset from Kaggle …")
kaggle_path = kagglehub.dataset_download("asmaaabdelwahab/arabic-news-dataset")
print(f"✅  Dataset path: {kaggle_path}")

# Auto-detect CSV files
csv_files = list(Path(kaggle_path).rglob("*.csv"))
print(f"   Found CSVs: {[f.name for f in csv_files]}")

# Load & concatenate all CSVs
dfs = [pd.read_csv(f) for f in csv_files]
df  = pd.concat(dfs, ignore_index=True)
print(f"   Total rows : {len(df)}")
print(f"   Columns    : {df.columns.tolist()}")

# ── Auto-detect article / summary columns ─────────────────────────────────────
def find_col(df, candidates):
    for c in candidates:
        for col in df.columns:
            if c.lower() in col.lower():
                return col
    return None

article_col = find_col(df, ["content", "article", "text", "body", "news"])
summary_col = find_col(df, ["summary", "title", "headline", "abstract"])

if article_col is None or summary_col is None:
    raise ValueError(
        f"Could not auto-detect columns. Available: {df.columns.tolist()}\n"
        "Set `article_col` and `summary_col` manually."
    )

print(f"   Article column : '{article_col}'")
print(f"   Summary column : '{summary_col}'")

# ── Clean & filter ────────────────────────────────────────────────────────────
df = df[[article_col, summary_col]].rename(
    columns={article_col: "article", summary_col: "summary"}
)
df = df.dropna().reset_index(drop=True)

def clean_text(t):
    t = str(t).strip()
    t = re.sub(r"\s+", " ", t)
    return t

df["article"] = df["article"].map(clean_text)
df["summary"] = df["summary"].map(clean_text)
df = df[(df["article"].str.len() > 50) & (df["summary"].str.len() > 10)].reset_index(drop=True)
print(f"   Rows after cleaning: {len(df)}")

# ── Train / Val / Test split (80 / 10 / 10) ──────────────────────────────────
train_df, temp_df = train_test_split(df, test_size=0.2,  random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
print(f"   Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

train_sum = HFDataset.from_pandas(train_df.reset_index(drop=True))
val_sum   = HFDataset.from_pandas(val_df.reset_index(drop=True))
test_sum  = HFDataset.from_pandas(test_df.reset_index(drop=True))

print("✅  Dataset ready.")

### 2.2 Preprocessing

In [ ]:
SUM_MODEL_NAME = "google/mt5-small"
MAX_INPUT_LEN  = 512
MAX_TARGET_LEN = 128

sum_tokenizer = AutoTokenizer.from_pretrained(SUM_MODEL_NAME)

PREFIX = "summarize: "   # task prefix for mT5

def preprocess_summarization(batch):
    inputs  = [PREFIX + art for art in batch["article"]]
    targets = batch["summary"]

    model_inputs = sum_tokenizer(
        inputs, max_length=MAX_INPUT_LEN,
        truncation=True, padding=False
    )
    with sum_tokenizer.as_target_tokenizer():
        labels = sum_tokenizer(
            targets, max_length=MAX_TARGET_LEN,
            truncation=True, padding=False
        )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train_sum = train_sum.map(
    preprocess_summarization, batched=True,
    remove_columns=train_sum.column_names
)
tokenized_val_sum = val_sum.map(
    preprocess_summarization, batched=True,
    remove_columns=val_sum.column_names
)
tokenized_test_sum = test_sum.map(
    preprocess_summarization, batched=True,
    remove_columns=test_sum.column_names
)

print("✅ Summarization dataset tokenized.")

### 2.3 Fine-Tune mT5

In [ ]:
from rouge_score import rouge_scorer as _rouge_scorer
from transformers import EarlyStoppingCallback

_scorer = _rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)

def compute_rouge(eval_pred):
    predictions, labels = eval_pred
    labels = np.where(labels != -100, labels, sum_tokenizer.pad_token_id)

    decoded_preds  = sum_tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = sum_tokenizer.batch_decode(labels,      skip_special_tokens=True)

    r1, r2, rL = [], [], []
    for pred, label in zip(decoded_preds, decoded_labels):
        scores = _scorer.score(label.strip(), pred.strip())
        r1.append(scores["rouge1"].fmeasure)
        r2.append(scores["rouge2"].fmeasure)
        rL.append(scores["rougeL"].fmeasure)

    return {
        "rouge1": round(np.mean(r1) * 100, 4),
        "rouge2": round(np.mean(r2) * 100, 4),
        "rougeL": round(np.mean(rL) * 100, 4),
    }


sum_model = AutoModelForSeq2SeqLM.from_pretrained(SUM_MODEL_NAME)

sum_collator = DataCollatorForSeq2Seq(
    tokenizer=sum_tokenizer,
    model=sum_model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

sum_training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5-arabic-summarization",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,      # effective batch = 32
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    num_train_epochs=5,
    fp16=torch.cuda.is_available(),
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rouge1",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    logging_steps=50,
    report_to=["tensorboard"],
    push_to_hub=False,
    dataloader_num_workers=2,
    seed=42,
)

sum_trainer = Seq2SeqTrainer(
    model=sum_model,
    args=sum_training_args,
    train_dataset=tokenized_train_sum,
    eval_dataset=tokenized_val_sum,
    tokenizer=sum_tokenizer,
    data_collator=sum_collator,
    compute_metrics=compute_rouge,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("🚀 Starting mT5 fine-tuning for Arabic summarization...")
sum_trainer.train()
sum_trainer.save_model("./mt5-arabic-summarization")
sum_tokenizer.save_pretrained("./mt5-arabic-summarization")
print("✅ Summarization model saved to ./mt5-arabic-summarization")

### 2.4 Evaluate on Test Set

In [ ]:
print("\n📊 Evaluating on held-out test set …")
test_results = sum_trainer.predict(tokenized_test_sum)
test_metrics = compute_rouge((test_results.predictions, test_results.label_ids))

print("\n" + "="*40)
print("       TEST SET ROUGE SCORES")
print("="*40)
for k, v in test_metrics.items():
    print(f"  {k:<10}: {v:.4f}")
print("="*40)

### 2.5 Inference Helper

In [ ]:
def summarize_arabic_text(text: str, model_path: str = "./mt5-arabic-summarization") -> str:
    """
    Summarize Arabic text using fine-tuned mT5.
    Handles long texts by chunking into 300-word blocks.
    """
    if os.path.exists(model_path):
        tok   = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(DEVICE)
        print(f"✅ Using fine-tuned summarizer: {model_path}")
    else:
        print("Fine-tuned model not found, using base mt5-small...")
        tok   = AutoTokenizer.from_pretrained("google/mt5-small")
        model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small").to(DEVICE)

    words    = text.split()
    chunks   = [" ".join(words[i:i+300]) for i in range(0, len(words), 300)]
    summaries = []

    for chunk in chunks:
        inputs = tok(
            "summarize: " + chunk, return_tensors="pt",
            max_length=512, truncation=True
        ).to(DEVICE)
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=128,
                num_beams=4, length_penalty=0.8,
                early_stopping=True, no_repeat_ngram_size=3
            )
        summaries.append(tok.decode(out[0], skip_special_tokens=True))

    return " ".join(summaries)


# ── Quick demo on one test sample ─────────────────────────────────────────────
if len(test_df) > 0:
    sample = test_df.iloc[0]
    gen    = summarize_arabic_text(sample["article"])
    print("── Inference Demo ──────────────────────────────")
    print(f"Article  : {sample['article'][:300]} …")
    print(f"Reference: {sample['summary']}")
    print(f"Generated: {gen}")

---
# 🔍 Task 3: Fine-Tune AraBERT for Arabic Semantic Search
We fine-tune `aubmindlab/bert-base-arabertv02` on **ARCD** (Arabic Reading Comprehension Dataset).

**Approach:** Contrastive learning — question–passage pairs are embedded close together.

**Evaluation Metrics:** Precision@K, Recall@K

### 3.1 Load ARCD Dataset

In [ ]:
# ARCD: Arabic Reading Comprehension Dataset
print("Loading ARCD dataset...")
arcd = load_dataset("hsseinmz/arcd", trust_remote_code=True)

print(f"Train: {len(arcd['train'])} | Validation: {len(arcd['validation'])}")
print("\nSample:")
s = arcd['train'][0]
print(f"Question : {s['question']}")
print(f"Context  : {s['context'][:200]}...")

### 3.2 Build Bi-Encoder Training Data (Contrastive Pairs)

In [ ]:
from sentence_transformers import InputExample, losses
from torch.utils.data import DataLoader

SEARCH_MODEL_NAME = "aubmindlab/bert-base-arabertv02"

# Build (question, passage) positive pairs from ARCD
def build_search_pairs(dataset_split, max_samples=5000):
    pairs = []
    for i, item in enumerate(dataset_split):
        if i >= max_samples:
            break
        question = item["question"]
        context  = item["context"]
        if question and context:
            pairs.append(InputExample(texts=[question, context], label=1.0))
    return pairs

train_pairs = build_search_pairs(arcd["train"], max_samples=8000)
print(f"✅ Training pairs: {len(train_pairs)}")
print(f"   Example Q: {train_pairs[0].texts[0]}")
print(f"   Example P: {train_pairs[0].texts[1][:100]}...")

### 3.3 Fine-Tune Bi-Encoder with Multiple Negatives Ranking Loss

In [ ]:
from sentence_transformers import SentenceTransformer, losses

# Load pre-trained Arabic BERT as bi-encoder
search_model = SentenceTransformer(SEARCH_MODEL_NAME, device=DEVICE)

train_dataloader = DataLoader(train_pairs, shuffle=True, batch_size=16)

# Multiple Negatives Ranking Loss — in-batch negatives (very effective for retrieval)
mnr_loss = losses.MultipleNegativesRankingLoss(model=search_model)

SEARCH_EPOCHS   = 3
WARMUP_STEPS    = int(len(train_dataloader) * SEARCH_EPOCHS * 0.1)

print(f"🚀 Fine-tuning Arabic bi-encoder for {SEARCH_EPOCHS} epochs...")
search_model.fit(
    train_objectives=[(train_dataloader, mnr_loss)],
    epochs=SEARCH_EPOCHS,
    warmup_steps=WARMUP_STEPS,
    output_path="./arabert-semantic-search",
    show_progress_bar=True
)
print("✅ Semantic search model saved to ./arabert-semantic-search")

### 3.4 Evaluate: Precision@K & Recall@K

In [ ]:
def evaluate_retrieval(model, eval_data, k=5, n_samples=500):
    """
    Evaluate retrieval quality using Precision@K and Recall@K.
    For each question, the correct passage should be in top-K results.
    """
    questions = []
    passages  = []
    for i, item in enumerate(eval_data):
        if i >= n_samples:
            break
        questions.append(item["question"])
        passages.append(item["context"])

    print(f"Encoding {len(questions)} questions and passages...")
    q_embeddings = model.encode(questions, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
    p_embeddings = model.encode(passages,  batch_size=32, show_progress_bar=True, normalize_embeddings=True)

    # Cosine similarity matrix
    cos_scores = util.cos_sim(q_embeddings, p_embeddings)  # (n_q, n_p)

    hits_at_k = 0
    for i in range(len(questions)):
        top_k_indices = cos_scores[i].topk(k).indices.tolist()
        if i in top_k_indices:   # correct passage has same index as question
            hits_at_k += 1

    precision_at_k = hits_at_k / len(questions)
    recall_at_k    = hits_at_k / len(questions)   # same for single-answer tasks

    print(f"\n📊 Retrieval Evaluation (K={k}, N={len(questions)} samples)")
    print(f"   Precision@{k} : {precision_at_k:.4f} ({hits_at_k}/{len(questions)})")
    print(f"   Recall@{k}    : {recall_at_k:.4f}")
    return precision_at_k, recall_at_k


# Load the fine-tuned model for evaluation
finetuned_search_model = SentenceTransformer("./arabert-semantic-search", device=DEVICE)
precision, recall = evaluate_retrieval(finetuned_search_model, arcd["validation"], k=5)

---
# 🔗 Full YouTube Pipeline: Audio → SRT Subtitles → Search
This combines all three fine-tuned models into a complete end-to-end system.

### 4.1 Transcribe Audio with Timestamps (Fine-tuned Whisper)

In [ ]:
def transcribe_with_timestamps(audio_path: str, model_path: str = "./whisper-arabic-finetuned") -> list:
    """
    Transcribe audio using fine-tuned Whisper.
    Returns list of {start, end, text} segments with word-level timestamps.
    """
    print(f"🎙️ Transcribing: {audio_path}")

    # Use fine-tuned model if available, otherwise fall back to base
    if os.path.exists(model_path):
        print(f"   Using fine-tuned model: {model_path}")
        model = whisper.load_model(model_path)
    else:
        print("   Fine-tuned model not found, using whisper-small...")
        model = whisper.load_model("small")

    result = model.transcribe(
        audio_path,
        language="ar",
        task="transcribe",
        word_timestamps=True,    # enables word-level alignment
        verbose=False
    )

    segments = []
    for seg in result["segments"]:
        segments.append({
            "start" : seg["start"],
            "end"   : seg["end"],
            "text"  : seg["text"].strip()
        })

    print(f"✅ Transcribed {len(segments)} segments.")
    return segments, result["text"]


# Run transcription
segments, full_transcript = transcribe_with_timestamps(audio_file)

# Preview first 5 segments
print("\n📋 First 5 segments:")
for s in segments[:5]:
    print(f"  [{s['start']:.1f}s → {s['end']:.1f}s] {s['text']}")

### 4.2 Generate SRT Subtitle File

In [ ]:
def seconds_to_srt_time(seconds: float) -> str:
    """Convert seconds to SRT timestamp format: HH:MM:SS,mmm"""
    td = timedelta(seconds=seconds)
    total_seconds = int(td.total_seconds())
    hours   = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    secs    = total_seconds % 60
    millis  = int((seconds - int(seconds)) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"


def generate_srt(segments: list, output_path: str = "subtitles.srt") -> str:
    """
    Generate an SRT subtitle file from Whisper segments.
    """
    srt_content = []
    for i, seg in enumerate(segments, start=1):
        start_time = seconds_to_srt_time(seg["start"])
        end_time   = seconds_to_srt_time(seg["end"])
        srt_content.append(f"{i}")
        srt_content.append(f"{start_time} --> {end_time}")
        srt_content.append(seg["text"])
        srt_content.append("")  # blank line between entries

    srt_text = "\n".join(srt_content)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(srt_text)
    print(f"✅ SRT subtitles saved to: {output_path}")
    return srt_text


srt_content = generate_srt(segments, output_path="arabic_subtitles.srt")
print("\n📄 SRT Preview (first 3 entries):")
print("\n".join(srt_content.split("\n")[:15]))

### 4.3 Summarize the Transcript (Fine-tuned mT5)

In [ ]:
def summarize_arabic_text(text: str, model_path: str = "./mt5-arabic-summarization") -> str:
    """
    Summarize Arabic text using fine-tuned mT5.
    Handles long texts by chunking.
    """
    if os.path.exists(model_path):
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(DEVICE)
        print(f"✅ Using fine-tuned summarizer: {model_path}")
    else:
        print("Fine-tuned model not found, using base mt5-small...")
        tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")
        model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small").to(DEVICE)

    # Chunk long transcripts (every ~400 tokens)
    words  = text.split()
    chunks = [" ".join(words[i:i+300]) for i in range(0, len(words), 300)]
    summaries = []

    for chunk in chunks:
        input_text = "summarize: " + chunk
        inputs = tokenizer(
            input_text, return_tensors="pt",
            max_length=512, truncation=True
        ).to(DEVICE)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=128,
                num_beams=4,
                early_stopping=True,
                no_repeat_ngram_size=3
            )
        summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        summaries.append(summary)

    return " ".join(summaries)


print("📝 Summarizing transcript...")
video_summary = summarize_arabic_text(full_transcript)
print("\n📋 Video Summary:")
print(video_summary)

### 4.4 Build FAISS Search Index Over Subtitle Segments

In [ ]:
class ArabicVideoSearchEngine:
    """
    Semantic search over Arabic video segments using FAISS.
    Enables queries like: 'متى تحدث عن الاقتصاد؟' (When did they talk about the economy?)
    """

    def __init__(self, model_path: str = "./arabert-semantic-search"):
        if os.path.exists(model_path):
            print(f"Loading fine-tuned search model: {model_path}")
            self.model = SentenceTransformer(model_path, device=DEVICE)
        else:
            print("Fine-tuned model not found, loading base AraBERT...")
            self.model = SentenceTransformer(
                "aubmindlab/bert-base-arabertv02", device=DEVICE
            )
        self.index    = None
        self.segments = []

    def build_index(self, segments: list):
        """Build FAISS index from transcript segments."""
        self.segments = segments
        texts = [s["text"] for s in segments]

        print(f"🔍 Encoding {len(texts)} segments...")
        embeddings = self.model.encode(
            texts, batch_size=32,
            show_progress_bar=True,
            normalize_embeddings=True
        )

        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)   # Inner product = cosine on normalized vecs
        self.index.add(embeddings.astype(np.float32))
        print(f"✅ FAISS index built with {self.index.ntotal} vectors (dim={dim}).")

    def search(self, query: str, top_k: int = 5) -> list:
        """Search for relevant segments given an Arabic query."""
        q_embedding = self.model.encode(
            [query], normalize_embeddings=True
        ).astype(np.float32)

        scores, indices = self.index.search(q_embedding, top_k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            seg = self.segments[idx]
            results.append({
                "score"     : float(score),
                "start"     : seg["start"],
                "end"       : seg["end"],
                "text"      : seg["text"],
                "timestamp" : f"{int(seg['start']//60):02d}:{int(seg['start']%60):02d}"
            })
        return results

    def save_index(self, path: str = "./search_index"):
        os.makedirs(path, exist_ok=True)
        faiss.write_index(self.index, f"{path}/faiss.index")
        with open(f"{path}/segments.json", "w", encoding="utf-8") as f:
            json.dump(self.segments, f, ensure_ascii=False, indent=2)
        print(f"✅ Search index saved to {path}/")

    def load_index(self, path: str = "./search_index"):
        self.index = faiss.read_index(f"{path}/faiss.index")
        with open(f"{path}/segments.json", encoding="utf-8") as f:
            self.segments = json.load(f)
        print(f"✅ Loaded search index: {self.index.ntotal} segments.")


# Build the search index
search_engine = ArabicVideoSearchEngine()
search_engine.build_index(segments)
search_engine.save_index()

### 4.5 Query the Search Engine

In [ ]:
def display_search_results(query: str, results: list):
    print(f"\n🔎 Query: '{query}'")
    print("=" * 60)
    for i, r in enumerate(results, 1):
        print(f"  #{i}  [{r['timestamp']}]  Score: {r['score']:.3f}")
        print(f"       {r['text']}")
        print()


# Example queries — change to match your video content
queries = [
    "ما هو الموضوع الرئيسي؟",        # What is the main topic?
    "متى ذكر الاقتصاد؟",              # When was the economy mentioned?
    "ما هي التوصيات المقترحة؟",       # What are the suggested recommendations?
]

for q in queries:
    results = search_engine.search(q, top_k=3)
    display_search_results(q, results)

---
## 📊 Evaluation Summary

In [ ]:
# ── Task 1: WER evaluation on test set ────────────────────────────
print("=" * 60)
print("📊 EVALUATION RESULTS")
print("=" * 60)

# Task 1 — ASR (WER)
print("\n[Task 1] Arabic ASR — Word Error Rate (WER)")
whisper_eval = trainer.evaluate()
print(f"   WER (fine-tuned Whisper): {whisper_eval['eval_wer']:.2f}%")

# Task 2 — Summarization (ROUGE)
print("\n[Task 2] Arabic Summarization — ROUGE Scores")
# test_metrics was computed in cell 2.4 above
print(f"   ROUGE-1 : {test_metrics['rouge1']:.4f}")
print(f"   ROUGE-2 : {test_metrics['rouge2']:.4f}")
print(f"   ROUGE-L : {test_metrics['rougeL']:.4f}")

# Task 3 — Search (Precision@K, Recall@K)
print("\n[Task 3] Arabic Semantic Search")
p5, r5 = evaluate_retrieval(finetuned_search_model, arcd["validation"], k=5)
p10, r10 = evaluate_retrieval(finetuned_search_model, arcd["validation"], k=10)
print(f"   Precision@5  : {p5:.4f}")
print(f"   Recall@5     : {r5:.4f}")
print(f"   Precision@10 : {p10:.4f}")
print(f"   Recall@10    : {r10:.4f}")

print("\n" + "=" * 60)
print("✅ All three tasks evaluated successfully!")

---
## 🎬 Demo: End-to-End on a New YouTube Video

In [ ]:
def process_arabic_youtube_video(url: str):
    """
    Full pipeline: YouTube URL → subtitles + summary + searchable index.
    """
    print("\n" + "="*60)
    print("🎬 ARABIC YOUTUBE PIPELINE")
    print("="*60)

    # Step 1: Download audio
    print("\n[1/4] Downloading audio...")
    audio_file = download_youtube_audio(url, output_path="temp_audio")

    # Step 2: Transcribe with timestamps
    print("\n[2/4] Transcribing with Whisper...")
    segments, full_text = transcribe_with_timestamps(audio_file)

    # Step 3: Generate SRT
    print("\n[3/4] Generating SRT subtitles...")
    srt = generate_srt(segments, output_path="output_subtitles.srt")

    # Step 4: Summarize
    print("\n[4/4] Summarizing transcript...")
    summary = summarize_arabic_text(full_text)

    # Build search index
    engine = ArabicVideoSearchEngine()
    engine.build_index(segments)
    engine.save_index(path="./output_search_index")

    # Cleanup temp file
    if os.path.exists(audio_file):
        os.remove(audio_file)

    print("\n" + "="*60)
    print("✅ PIPELINE COMPLETE")
    print("="*60)
    print(f"  📄 SRT file   : output_subtitles.srt")
    print(f"  📋 Summary    : {summary[:200]}...")
    print(f"  🔍 Search idx : ./output_search_index/")
    print(f"  📦 Segments   : {len(segments)}")

    return {
        "segments" : segments,
        "full_text": full_text,
        "summary"  : summary,
        "srt"      : srt,
        "engine"   : engine
    }


# ── Run the full pipeline ──────────────────────────────────────────
DEMO_URL = "https://www.youtube.com/watch?v=YOUR_VIDEO_ID"  # ← change this
output = process_arabic_youtube_video(DEMO_URL)

# Interactive search
query  = "اذكر أهم النقاط"   # 'mention the main points'
hits   = output["engine"].search(query, top_k=3)
display_search_results(query, hits)